In [1]:
!pip install datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.2/401.2 kB 8.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.20.3
    Uninstalling huggingface-hub-0.20.3:
      Successfully uninstalled huggingface-hub-0.20.3


In [2]:
import pandas as pd
import numpy as np
import keras
from keras.preprocessing.text import Tokenizer
from keras.utils import pad_sequences
from keras.models import Sequential, Model
from keras.layers import Embedding, Dense, Bidirectional, LSTM, SimpleRNN, Flatten, Input, Concatenate, Attention
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer

In [3]:
from datasets import load_dataset

dataset = load_dataset("rajpurkar/squad")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

In [5]:
type(dataset)

datasets.dataset_dict.DatasetDict

In [6]:
train= dataset["train"]

for i in range(5):
    example = train[i]
    print("Example", i+1)
    print("ID:", example['id'])
    print("Title:", example['title'])
    print("Context:", example['context'])
    print("Question:", example['question'])
    print("Answers:", example['answers'])
    print()

Example 1
ID: 5733be284776f41900661182
Title: University_of_Notre_Dame
Context: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.
Question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Answers: {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}

Example 2
ID: 5733be284776f4190066117f
Title: University_of_Notre_Da

In [7]:
validation= dataset["validation"]

for i in range(5):
    example = validation[i]
    print("Example", i+1)
    print("ID:", example['id'])
    print("Title:", example['title'])
    print("Context:", example['context'])
    print("Question:", example['question'])
    print("Answers:", example['answers'])
    print()

Example 1
ID: 56be4db0acb8001400a502ec
Title: Super_Bowl_50
Context: Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.
Question: Which NFL team represented the AFC at Super Bowl 50?
Answers: {'text': ['Denver Broncos', 'Denver Broncos', 'Denver Broncos'], 'answer_start': [1

In [8]:
questions_train = [item['question'] for item in train]
answers_train = [item['answers']['text'][0] for item in train]

questions_test = [item['question'] for item in validation]
answers_test = [item['answers']['text'][0] for item in validation]

In [9]:
print(questions_train[0:5])
print(answers_train[0:5])

print(questions_test[0:5])
print(answers_test[0:5])

['To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?', 'What is in front of the Notre Dame Main Building?', 'The Basilica of the Sacred heart at Notre Dame is beside to which structure?', 'What is the Grotto at Notre Dame?', 'What sits on top of the Main Building at Notre Dame?']
['Saint Bernadette Soubirous', 'a copper statue of Christ', 'the Main Building', 'a Marian place of prayer and reflection', 'a golden statue of the Virgin Mary']
['Which NFL team represented the AFC at Super Bowl 50?', 'Which NFL team represented the NFC at Super Bowl 50?', 'Where did Super Bowl 50 take place?', 'Which NFL team won Super Bowl 50?', 'What color was used to emphasize the 50th anniversary of the Super Bowl?']
['Denver Broncos', 'Carolina Panthers', 'Santa Clara, California', 'Denver Broncos', 'gold']


In [10]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(questions_train + answers_train + questions_test + answers_test)

questions_train_seq = tokenizer.texts_to_sequences(questions_train)
questions_test_seq = tokenizer.texts_to_sequences(questions_test)

answer_train_seq = tokenizer.texts_to_sequences(answers_train)
answer_test_seq = tokenizer.texts_to_sequences(answers_test)

vocab_size = len(tokenizer.word_index) + 1

In [11]:
print(vocab_size)

59933


In [12]:
questions_train_seq[0:5]

[[5, 304, 8, 1, 2407, 739, 7240, 842, 4, 5726, 4, 19534, 178],
 [2, 7, 4, 1545, 3, 1, 617, 611, 167, 247],
 [1, 3685, 3, 1, 3272, 1594, 33, 617, 611, 7, 6139, 5, 15, 761],
 [2, 7, 1, 24399, 33, 617, 611],
 [2, 9409, 18, 388, 3, 1, 167, 247, 33, 617, 611]]

In [13]:
questions_test_seq[0:5]

[[15, 2172, 200, 1705, 1, 8373, 33, 252, 263, 321],
 [15, 2172, 200, 1705, 1, 6138, 33, 252, 263, 321],
 [26, 8, 252, 263, 321, 122, 131],
 [15, 2172, 200, 459, 252, 263, 321],
 [2, 347, 6, 38, 5, 8122, 1, 6046, 2655, 3, 1, 252, 263]]

In [14]:
maxlen_Qtrain = max([len(seq) for seq in questions_train_seq])
maxlen_Qtest = max([len(seq) for seq in questions_test_seq])

maxlen_Atrain = max([len(seq) for seq in answer_train_seq])
maxlen_Atest = max([len(seq) for seq in answer_test_seq])

In [15]:
print(maxlen_Qtrain)
print(maxlen_Atrain)

print(maxlen_Qtest)
print(maxlen_Atest)

40
43
33
30


In [16]:
max_seq_len = max(maxlen_Qtrain, maxlen_Atrain, maxlen_Qtest, maxlen_Atest)

embd_size= 100
train_questions_seq= pad_sequences(questions_train_seq, maxlen= max_seq_len, padding ='post', truncating='post')
test_questions_seq= pad_sequences(questions_test_seq, maxlen= max_seq_len, padding = 'post', truncating='post')

train_answers_seq= pad_sequences(answer_train_seq, maxlen= max_seq_len, padding ='post', truncating='post')
test_answers_seq= pad_sequences(answer_test_seq, maxlen= max_seq_len, padding = 'post', truncating='post')

In [17]:
print(train_questions_seq.shape)
print(test_questions_seq.shape)
print(train_answers_seq.shape)
print(test_answers_seq.shape)

(87599, 43)
(10570, 43)
(87599, 43)
(10570, 43)


In [18]:
from keras.layers import Input, Embedding, LSTM, Dense, Bidirectional, Concatenate, Dropout
from keras.models import Model
from keras.optimizers import Adam

def build_model(vocab_size, maxlen_questions, maxlen_answers):
    # Encoder
    question_input = Input(shape=(maxlen_questions,), name='question_input')
    embedding_layer = Embedding(input_dim=vocab_size, output_dim=128, mask_zero=True)
    question_embedded = embedding_layer(question_input)
    encoder_lstm = Bidirectional(LSTM(64, return_state=True, dropout=0.5, recurrent_dropout=0.5))
    question_encoded, forward_h, forward_c, backward_h, backward_c = encoder_lstm(question_embedded)
    state_h = Concatenate()([forward_h, backward_h])
    state_c = Concatenate()([forward_c, backward_c])

    # Decoder
    answer_input = Input(shape=(maxlen_answers,), name='answer_input')
    answer_embedded = embedding_layer(answer_input)
    decoder_lstm = LSTM(128, return_sequences=True, return_state=True, dropout=0.5, recurrent_dropout=0.5)
    answer_encoded, _, _ = decoder_lstm(answer_embedded, initial_state=[state_h, state_c])

    # Output
    dense = Dense(128, activation='relu')(answer_encoded)
    output_layer = Dense(vocab_size, activation='softmax')(dense)

    # Model
    model = Model(inputs=[question_input, answer_input], outputs=output_layer)
    model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    return model

model = build_model(vocab_size, max_seq_len, max_seq_len)
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 answer_input (InputLayer)   [(None, 43)]                 0         []                            
                                                                                                  
 question_input (InputLayer  [(None, 43)]                 0         []                            
 )                                                                                                
                                                                                                  
 embedding (Embedding)       (None, 43, 128)              7671424   ['question_input[0][0]',      
                                                                     'answer_input[0][0]']        
                                                                                              

In [19]:
train_answers_seq

array([[  499, 44482, 44483, ...,     0,     0,     0],
       [   10,   791,  4620, ...,     0,     0,     0],
       [    1,   167,   247, ...,     0,     0,     0],
       ...,
       [57226,     0,     0, ...,     0,     0,     0],
       [ 2639,     0,     0, ...,     0,     0,     0],
       [ 1621,  1490,    41, ...,     0,     0,     0]], dtype=int32)

In [20]:
model.fit([train_questions_seq, train_answers_seq], train_answers_seq,
          epochs=1,
          validation_data=([test_questions_seq, test_answers_seq], test_answers_seq),
          batch_size=32)

2738/2738 [==============================] - 11915s 4s/step - loss: 6.2645 - accuracy: 0.2385 - val_loss: 4.9206 - val_accuracy: 0.4406


In [23]:
def predict_answer(question, model, tokenizer, max_len):
    question_seq = tokenizer.texts_to_sequences([question])
    question_seq = pad_sequences(question_seq, maxlen=max_len, padding='post')
    answer_seq = np.zeros((1, max_len))
    answer_output = model.predict([question_seq, answer_seq])
    answer_index = np.argmax(answer_output[0])
    answer = tokenizer.sequences_to_texts([[answer_index]])[0]
    return answer

test_question = "Where did Super Bowl 50 take place?"
predicted_answer = predict_answer(test_question, model, tokenizer, max_seq_len)
print("Predicted Answer:", predicted_answer)

1/1 [==============================] - 0s 122ms/step
Predicted Answer: muhammad
